In [ ]:
import os
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from sklearn.metrics import (classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score,
                             roc_curve, precision_recall_curve, average_precision_score, accuracy_score, f1_score)
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.preprocessing import label_binarize
from lightgbm import LGBMClassifier
from matplotlib.backends.backend_pdf import PdfPages
import warnings

warnings.filterwarnings("ignore")

# === CONFIGURACIÓN ===
INPUT_PATH = r"C:\Users\Gonzalo\Documents\GitHub\TesisAustral\Archivos_tesis\intermediate\selected"
BASE_OUTPUT_PATH = r"C:\Users\Gonzalo\Documents\GitHub\TesisAustral\Archivos_tesis\models"
MODEL_NAME = "LightGBM"
INTENTO = 2  # Cambiar manualmente

# Crear carpeta de salida
fecha_actual = datetime.today().strftime('%Y-%m-%d')
OUTPUT_PATH = os.path.join(BASE_OUTPUT_PATH, f"{MODEL_NAME}_{INTENTO:02d}_{fecha_actual}")
os.makedirs(OUTPUT_PATH, exist_ok=True)

resumen_global = []

# Hiperparámetros a optimizar
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 6, 10],
    'learning_rate': [0.01, 0.1],
    'class_weight': ['balanced', None]
}

for file in os.listdir(INPUT_PATH):
    if not file.startswith("X_train") or not file.endswith(".parquet"):
        continue

    nombre_base = file.replace("X_train_", "").replace(".parquet", "")
    print(f"\n🔍 Procesando: {nombre_base}")
    try:
        # Cargar datos
        X_train = pd.read_parquet(os.path.join(INPUT_PATH, f"X_train_{nombre_base}.parquet"))
        y_train = pd.read_parquet(os.path.join(INPUT_PATH, f"y_train_{nombre_base}.parquet")).squeeze()
        X_test = pd.read_parquet(os.path.join(INPUT_PATH, f"X_test_{nombre_base}.parquet"))
        y_test = pd.read_parquet(os.path.join(INPUT_PATH, f"y_test_{nombre_base}.parquet")).squeeze()

        # Validación cruzada con GridSearch
        start_time = time.time()
        model = LGBMClassifier(random_state=42, n_jobs=-1)
        grid = GridSearchCV(model, param_grid, cv=StratifiedKFold(n_splits=3),
                            scoring='f1_weighted', n_jobs=-1, verbose=0)
        grid.fit(X_train, y_train)
        best_model = grid.best_estimator_
        fit_time = time.time() - start_time

        # Evaluación
        y_pred_test = best_model.predict(X_test)
        y_proba_test = best_model.predict_proba(X_test)
        y_pred_train = best_model.predict(X_train)

        report_test = classification_report(y_test, y_pred_test, output_dict=True)
        report_train = classification_report(y_train, y_pred_train, output_dict=True)

        macro_f1_test = report_test['macro avg']['f1-score']
        macro_f1_train = report_train['macro avg']['f1-score']
        delta_macro_f1 = macro_f1_train - macro_f1_test

        # Matriz de confusión
        cm = confusion_matrix(y_test, y_pred_test)
        pdf_path = os.path.join(OUTPUT_PATH, f"reporte_{nombre_base}.pdf")
        with PdfPages(pdf_path) as pdf:
            ConfusionMatrixDisplay(cm).plot(cmap='Blues')
            plt.title("Matriz de Confusión")
            pdf.savefig()
            plt.close()

            # ROC
            classes = np.unique(y_test)
            y_test_bin = label_binarize(y_test, classes=classes)
            plt.figure()
            for i, clase in enumerate(classes):
                fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_proba_test[:, i])
                auc_roc = roc_auc_score(y_test_bin[:, i], y_proba_test[:, i])
                plt.plot(fpr, tpr, label=f"Clase {clase} (AUC={auc_roc:.2f})")
            plt.plot([0, 1], [0, 1], 'k--')
            plt.title("Curvas ROC")
            plt.xlabel("FPR")
            plt.ylabel("TPR")
            plt.legend()
            pdf.savefig()
            plt.close()

            # Precision-Recall
            plt.figure()
            for i, clase in enumerate(classes):
                prec, rec, _ = precision_recall_curve(y_test_bin[:, i], y_proba_test[:, i])
                pr_auc = average_precision_score(y_test_bin[:, i], y_proba_test[:, i])
                plt.plot(rec, prec, label=f"Clase {clase} (PR AUC={pr_auc:.2f})")
            plt.title("Curvas Precision-Recall")
            plt.xlabel("Recall")
            plt.ylabel("Precision")
            plt.legend()
            pdf.savefig()
            plt.close()

        # Guardar CSV
        df_metrics = pd.DataFrame(report_test).transpose()
        df_metrics['macro_f1_train'] = macro_f1_train
        df_metrics['macro_f1_test'] = macro_f1_test
        df_metrics['delta_macro_f1'] = delta_macro_f1
        df_metrics['fit_minutes'] = round(fit_time / 60, 2)
        df_metrics.to_csv(os.path.join(OUTPUT_PATH, f"metricas_{nombre_base}.csv"))

        resumen_global.append({
            "modelo": nombre_base,
            "macro_f1_train": macro_f1_train,
            "macro_f1_test": macro_f1_test,
            "delta_macro_f1": delta_macro_f1,
            "accuracy_test": accuracy_score(y_test, y_pred_test),
            "f1_weighted_test": f1_score(y_test, y_pred_test, average='weighted'),
            "fit_minutes": round(fit_time / 60, 2),
            "mejores_parametros": grid.best_params_
        })

        print(f"✅ {nombre_base} terminado. Macro F1 Test: {macro_f1_test:.3f}, Overfitting Delta: {delta_macro_f1:.3f}")

    except Exception as e:
        print(f"❌ Error en {nombre_base}: {e}")

# CSV Global
pd.DataFrame(resumen_global).to_csv(os.path.join(OUTPUT_PATH, "resumen_modelos_lightgbm.csv"), index=False)
print("\n📊 Todos los modelos procesados. CSV global guardado.")



🔍 Procesando: MaxAbs_FE_ALL
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.832484 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 18843
[LightGBM] [Info] Number of data points in the train set: 448388, number of used features: 967
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Info] Start training from score -1.609438
✅ MaxAbs_FE_ALL terminado. Macro F1 Test: 0.431, Overfitting Delta: 0.051

🔍 Procesando: MaxAbs_FE_ANOVA
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.206901 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 8647
[LightGBM] [Info] Number of data points in t